# Data control panel

Local control panel for checking data freshness, updating the tournament master file, and running the project smoke check.

In [ ]:
from __future__ import annotations

import html as html_module
import os
import subprocess
import sys
import threading
from pathlib import Path

from IPython.display import Markdown, display
import ipywidgets as widgets


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not find project root.')


def resolve_project_python(project_root: Path) -> str:
    candidates = [
        project_root / '.venv' / 'Scripts' / 'python.exe',
        project_root / 'venv' / 'Scripts' / 'python.exe',
        project_root / '.venv' / 'bin' / 'python',
        project_root / 'venv' / 'bin' / 'python',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return sys.executable


PROJECT_ROOT = find_project_root()
PYTHON = resolve_project_python(PROJECT_ROOT)
BUTTONS: list[widgets.Button] = []
LOG_LINES: list[str] = []
MAX_LOG_LINES = 1000


def set_buttons_disabled(disabled: bool) -> None:
    for button in BUTTONS:
        button.disabled = disabled


def set_log(text: str) -> None:
    log_area.value = text


def append_log(text: str) -> None:
    for line in text.splitlines() or ['']:
        LOG_LINES.append(line)
    del LOG_LINES[:-MAX_LOG_LINES]
    log_area.value = '\n'.join(LOG_LINES)


def describe_output_line(line: str, title: str) -> str | None:
    text = line.strip()
    if not text:
        return None
    if text.startswith('$ '):
        return 'Launching command'
    if text.startswith('===') and text.endswith('==='):
        return text.strip('= ').strip()
    if text.startswith('Full RTT predictor pipeline'):
        return 'Full pipeline started'
    if text.startswith('Preflight dependency check'):
        return 'Checking Python dependencies'
    if text.startswith('Missing Python packages:'):
        return text
    if text.startswith('Python executable:'):
        return text
    if text.startswith('Project root:'):
        return 'Checking project location'
    if text.startswith('Started at:'):
        return 'Preparing pipeline steps'
    if text.startswith('Opening RTT calendar'):
        return text
    if text.startswith('Warning: RTT calendar update failed'):
        return text
    if text.startswith('Parsing calendar age group:'):
        return 'Calendar filter: ' + text.split(':', 1)[1].strip()
    if text.startswith('Fetching tournament details'):
        return text
    if text.startswith('Tournament download plan:'):
        return text
    if text.startswith('future_start_date') or text.startswith('cached_html') or text.startswith('completed_cached') or text.startswith('cancelled'):
        return 'Skipping/caching: ' + text
    if text.startswith('Updated matches_page_saved'):
        return text
    if text.startswith('Parsed tournaments:'):
        return text
    if text.startswith('Master before:') or text.startswith('Master after:'):
        return text
    if text.startswith('New tournaments:'):
        return text
    if text.startswith('Output:'):
        return 'Writing output file: ' + text.split(':', 1)[1].strip()
    if text.startswith('Data status'):
        return 'Reading local data status'
    if text.startswith('Rankings rows:'):
        return 'Checking rankings dataset'
    if text.startswith('Matches rows:'):
        return 'Checking parsed matches dataset'
    if text.startswith('Dataset rows:'):
        return 'Checking model dataset'
    if text.startswith('Wrote') and 'data_manifest' in text:
        return 'Updating data manifest'
    if text.startswith('notebook ok:'):
        return 'Verifying notebook: ' + text.split(':', 1)[1].strip()
    if text.startswith('dataset ok'):
        return 'Verifying final dataset'
    if text.startswith('tournaments ok'):
        return 'Verifying tournament master'
    if text.startswith('project verification passed'):
        return 'Project verification passed'
    if text.startswith('Step failed:'):
        return text
    if 'ModuleNotFoundError' in text:
        return text
    return None


def set_current_step(text: str) -> None:
    current_step_html.value = '<b>Now:</b> ' + html_module.escape(text)


def run_project_command_streaming(title: str, args: list[str]) -> None:
    set_buttons_disabled(True)
    progress.value = 0
    progress.bar_style = 'info'
    status_html.value = f'<b>{title}</b>: running...'
    set_current_step('Starting ' + title)

    command = [PYTHON, '-u', *args]
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONUTF8'] = '1'

    LOG_LINES.clear()
    set_log('')
    append_log(f'### {title}')
    append_log('$ ' + ' '.join(command))
    append_log('')

    try:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )

        line_count = 0
        assert process.stdout is not None
        for line in process.stdout:
            line_count += 1
            progress.value = min(99, 5 + line_count)
            current_description = describe_output_line(line, title)
            if current_description:
                set_current_step(current_description)
            append_log(line.rstrip('\n'))

        return_code = process.wait()
        if return_code == 0:
            progress.value = 100
            progress.bar_style = 'success'
            status_html.value = f'<b>{title}</b>: done'
            set_current_step('Completed ' + title)
        else:
            progress.value = 100
            progress.bar_style = 'danger'
            status_html.value = f'<b>{title}</b>: failed with code {return_code}'
            set_current_step(f'Stopped after error code {return_code}')
    except Exception as exc:
        progress.value = 100
        progress.bar_style = 'danger'
        status_html.value = f'<b>{title}</b>: failed'
        set_current_step('Stopped after an unexpected error')
        append_log(str(exc))
    finally:
        set_buttons_disabled(False)


def show_command(title: str, args: list[str]) -> None:
    thread = threading.Thread(target=run_project_command_streaming, args=(title, args), daemon=True)
    thread.start()


status_button = widgets.Button(description='Refresh status', button_style='info')
tournaments_button = widgets.Button(description='Update tournaments master', button_style='warning')
calendar_button = widgets.Button(description='Update RTT calendar', button_style='warning')
verify_button = widgets.Button(description='Verify project', button_style='success')
pipeline_button = widgets.Button(description='Run full pipeline', button_style='danger')
BUTTONS.extend([status_button, tournaments_button, calendar_button, pipeline_button, verify_button])

progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='')
status_html = widgets.HTML(value='Ready')
current_step_html = widgets.HTML(value='<b>Now:</b> idle')
log_area = widgets.Textarea(
    value='',
    disabled=True,
    layout=widgets.Layout(width='100%', height='360px'),
)

status_button.on_click(lambda _: show_command('Data status', ['scripts/data_status.py', '--write-manifest']))
tournaments_button.on_click(lambda _: show_command('Tournament master update', ['scripts/bootstrap_tournaments_master.py']))
calendar_button.on_click(lambda _: show_command('RTT calendar update', ['scripts/parse_rtt_calendar.py']))
pipeline_button.on_click(lambda _: show_command('Full update pipeline', ['scripts/run_full_pipeline.py', '--continue-on-calendar-error']))
verify_button.on_click(lambda _: show_command('Project verification', ['scripts/verify_project.py']))

display(Markdown(f'Project root: `{PROJECT_ROOT}`'))
display(Markdown(f'Python: `{PYTHON}`'))
display(widgets.HBox([status_button, tournaments_button, calendar_button, pipeline_button, verify_button]))
display(widgets.VBox([status_html, current_step_html, progress, log_area]))
show_command('Data status', ['scripts/data_status.py', '--write-manifest'])


In [ ]:
# Unified prediction command center
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts import prediction_runtime as pr

prediction_output = widgets.Output()

try:
    prediction_bundle = pr.load_prediction_bundle()
    player_names = pr.player_options(prediction_bundle)
    max_match_date = pd.Timestamp(prediction_bundle['long_feat']['match_date'].max()).date()
    bundle_status = f"Loaded prediction bundle: {Path(prediction_bundle['bundle_path']).name}; max match date: {max_match_date}"
except Exception as exc:
    prediction_bundle = None
    player_names = []
    max_match_date = pd.Timestamp.today().date()
    bundle_status = f"Prediction bundle is not ready: {exc}"

player1_box = widgets.Combobox(
    placeholder='Start typing player 1 surname',
    options=player_names,
    description='Player 1:',
    ensure_option=False,
    layout=widgets.Layout(width='48%'),
)
player2_box = widgets.Combobox(
    placeholder='Start typing player 2 surname',
    options=player_names,
    description='Player 2:',
    ensure_option=False,
    layout=widgets.Layout(width='48%'),
)
prediction_date_picker = widgets.DatePicker(
    description='Date:',
    value=max_match_date,
    layout=widgets.Layout(width='220px'),
)
age_box = widgets.Dropdown(
    options=['__UNKNOWN_AGE__', 'U15', 'U17', 'U19', 'adult'],
    value='__UNKNOWN_AGE__',
    description='Age:',
    layout=widgets.Layout(width='240px'),
)
predict_button = widgets.Button(description='Predict match', button_style='primary')
reload_bundle_button = widgets.Button(description='Reload bundle', button_style='info')
prediction_status = widgets.HTML(value=f"<b>Prediction:</b> {bundle_status}")


def _probability_bar(player1_name, player2_name, p1):
    p2 = 1.0 - p1
    return f'''
    <div style="font-family: system-ui; max-width: 980px;">
      <h3 style="margin-bottom: 6px;">{player1_name} vs {player2_name}</h3>
      <div style="font-size: 18px; margin-bottom: 8px;">
        <b>{player1_name}</b>: {p1:.1%} &nbsp; | &nbsp; <b>{player2_name}</b>: {p2:.1%}
      </div>
      <div style="height: 22px; border: 1px solid #ddd; border-radius: 4px; overflow: hidden; display: flex;">
        <div style="width: {p1*100:.2f}%; background: #4f7cff;"></div>
        <div style="width: {p2*100:.2f}%; background: #ff8a4f;"></div>
      </div>
    </div>
    '''


def _display_profiles(result):
    profiles = pr.profiles_table(result).copy()
    for col in ['Win rate', 'Form 5', 'Form 10']:
        if col in profiles.columns:
            profiles[col] = profiles[col].map(lambda x: f'{x:.1%}' if pd.notna(x) else '')
    display(HTML('<h4>Player profiles</h4>'))
    display(profiles)


def _plot_probability_timeline(bundle, player1_name, player2_name, prediction_date):
    timeline = pr.probability_timeline(bundle, player1_name, player2_name, prediction_date)
    if timeline.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(timeline['date'], timeline['p_player1_win'], marker='o', linewidth=2)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_title(f'Historical win probability: {player1_name}')
    ax.set_ylabel('Probability')
    ax.grid(alpha=0.25)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def _plot_player_history(bundle, player_id, title):
    history = pr.player_history(bundle, player_id, pd.Timestamp.max)
    if history.empty:
        return
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    if 'player_points_pre' in history.columns:
        axes[0].plot(history['match_date'], history['player_points_pre'], marker='o', linewidth=1.5)
    axes[0].set_title(f'{title}: rating points')
    axes[0].grid(alpha=0.25)
    if 'elo_pre' in history.columns:
        axes[1].plot(history['match_date'], history['elo_pre'], marker='o', linewidth=1.5, color='#3a9d62')
    axes[1].set_title(f'{title}: ELO')
    axes[1].grid(alpha=0.25)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def _display_factor_contributions(result):
    contrib = result.get('factor_contributions', pd.DataFrame()).copy()
    if contrib.empty:
        return
    display(HTML('<h4>Local factor contribution to player 1 probability</h4>'))
    show_cols = ['feature', 'group', 'probability_effect', 'value_player1_row', 'value_player2_row', 'meaning']
    show_cols = [col for col in show_cols if col in contrib.columns]
    table = contrib[show_cols].copy()
    table['probability_effect'] = table['probability_effect'].map(lambda x: f'{x:+.1%}')
    display(table)

    plot_df = contrib.sort_values('probability_effect')
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_df))))
    colors = ['#ff8a4f' if v < 0 else '#4f7cff' for v in plot_df['probability_effect']]
    ax.barh(plot_df['feature'], plot_df['probability_effect'], color=colors)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Probability shift for player 1')
    ax.set_title('Main forecast drivers')
    ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def run_prediction(_=None):
    with prediction_output:
        clear_output(wait=True)
        if prediction_bundle is None:
            display(HTML('<b>Prediction bundle is missing.</b> Run full pipeline or train model first.'))
            return
        if not player1_box.value or not player2_box.value:
            display(HTML('<b>Select two players.</b>'))
            return
        if player1_box.value == player2_box.value:
            display(HTML('<b>Players must be different.</b>'))
            return

        context = {
            'tournament_age_category': age_box.value,
            'draw_type': '__UNKNOWN_DRAW__',
            'tournament_name': '__USER_PREDICTION__',
            'tournament_city': '__UNKNOWN_CITY__',
        }
        result = pr.predict_match_by_names(
            prediction_bundle,
            player1_box.value,
            player2_box.value,
            pd.Timestamp(prediction_date_picker.value),
            context=context,
        )
        if not result.get('ok'):
            display(HTML(f"<b>{result.get('message')}</b>"))
            display(result.get('player1_lookup', {}).get('candidates', pd.DataFrame()))
            display(result.get('player2_lookup', {}).get('candidates', pd.DataFrame()))
            return

        p1_name = result['player1_lookup']['player_name']
        p2_name = result['player2_lookup']['player_name']
        display(HTML(_probability_bar(p1_name, p2_name, result['p_player1_win'])))
        display(HTML(f"<p><b>Model:</b> {result['model_name']}</p>"))
        _display_profiles(result)
        _plot_probability_timeline(prediction_bundle, p1_name, p2_name, pd.Timestamp(prediction_date_picker.value))
        _plot_player_history(prediction_bundle, result['player1_lookup']['player_id'], p1_name)
        _plot_player_history(prediction_bundle, result['player2_lookup']['player_id'], p2_name)
        _display_factor_contributions(result)
        display(HTML('<h4>Prediction rows</h4>'))
        display(result['prediction_rows'].T)


def reload_prediction_bundle(_=None):
    global prediction_bundle, player_names
    with prediction_output:
        clear_output(wait=True)
        try:
            prediction_bundle = pr.load_prediction_bundle()
            player_names = pr.player_options(prediction_bundle)
            player1_box.options = player_names
            player2_box.options = player_names
            prediction_status.value = f"<b>Prediction:</b> Loaded {Path(prediction_bundle['bundle_path']).name}; max match date: {pd.Timestamp(prediction_bundle['long_feat']['match_date'].max()).date()}"
            display(HTML('<b>Prediction bundle reloaded.</b>'))
        except Exception as exc:
            display(HTML(f'<b>Could not reload prediction bundle:</b> {exc}'))


predict_button.on_click(run_prediction)
reload_bundle_button.on_click(reload_prediction_bundle)

prediction_panel = widgets.VBox([
    widgets.HTML('<h3>Match forecast and analytics</h3>'),
    prediction_status,
    widgets.HBox([player1_box, player2_box]),
    widgets.HBox([prediction_date_picker, age_box, predict_button, reload_bundle_button]),
    prediction_output,
])

display(prediction_panel)
